In [0]:
# Data Cleaning -- mirrors /Users/rajendramoka7@gmail.com/DE/datacleaning
# Read the raw employee data (update the path if the CSV lives elsewhere, e.g. /Volumes/.../Employee_Table.csv)
df = spark.read.csv(
    'Employee_Table.csv',
    header=True,
    inferSchema=True
)

df.printSchema()
print('Before Data Cleaning')
display(df)

# Fill null values in Department with 'Generalist'
df = df.na.fill('Generalist', subset=['Department'])

# Assumed null Joining Year means the employee joined during company founding (2010)
df = df.na.fill(2010, subset=['Joining Year'])

# Replace 'Information Tech' with 'IT'
df = df.replace({'Information Tech': 'IT'}, subset=['Department'])

# Remove outliers: valid working age is between 19 and 59
df = df.filter((df['Age'] < 60) & (df['Age'] > 18))

# Remove rows missing both First and Last Name
df = df.filter(df['First Name'].isNotNull() | df['Last Name'].isNotNull())

print('After Data Cleaning')
display(df)

## Advanced Data Transformations
Building on the cleaned data from the cell above:

1. **Text standardization** – trim, casing, and regex cleanup
2. **Derived columns** – Full Name, Age Group, Years of Service, Tenure Category
3. **Aggregations & pivot** – per-department stats and Tenure Category breakdown
4. **Window functions** – ranking, department averages broadcast back to rows
5. **Date handling** – real dates from Joining Year and veteran filtering

In [0]:
from pyspark.sql import functions as F

# Standardize text columns: trim whitespace, fix casing, collapse repeated spaces
df_std = (
    df
    .withColumn('First Name', F.initcap(F.trim('First Name')))
    .withColumn('Last Name', F.initcap(F.trim('Last Name')))
    .withColumn('Department', F.upper(F.regexp_replace(F.trim('Department'), r'\s+', ' ')))
)

display(df_std)

In [0]:
# Derived columns using native expressions (when/otherwise instead of UDFs)
df_adv = (
    df_std
    # Full Name: concat_ws skips null name parts instead of returning null
    .withColumn('Full Name', F.concat_ws(' ', 'First Name', 'Last Name'))
    # Years of Service: tenure derived from Joining Year
    .withColumn('Years of Service', F.year(F.current_date()) - F.col('Joining Year'))
    # Age Group buckets
    .withColumn(
        'Age Group',
        F.when(F.col('Age') < 30, 'Junior')
         .when((F.col('Age') >= 30) & (F.col('Age') < 45), 'Mid-level')
         .when((F.col('Age') >= 45) & (F.col('Age') < 60), 'Senior')
         .otherwise('Unknown')
    )
    # Tenure Category from Years of Service
    .withColumn(
        'Tenure Category',
        F.when(F.col('Years of Service') >= 10, 'Veteran')
         .when(F.col('Years of Service') >= 5, 'Established')
         .otherwise('New Joiner')
    )
)

display(df_adv)

In [0]:
# GroupBy aggregations per department
dept_stats = (
    df_adv.groupBy('Department')
    .agg(
        F.count('*').alias('Employee Count'),
        F.round(F.avg('Age'), 1).alias('Avg Age'),
        F.round(F.avg('Years of Service'), 1).alias('Avg Tenure'),
        F.min('Joining Year').alias('Earliest Joining Year'),
        F.max('Joining Year').alias('Latest Joining Year'),
    )
    .orderBy(F.desc('Employee Count'))
)
display(dept_stats)

# Pivot: employee count per department split by Tenure Category
pivot_df = df_adv.groupBy('Department').pivot('Tenure Category').count()
display(pivot_df)

In [0]:
from pyspark.sql.window import Window

# Rank employees by age within each department (youngest first)
w_dept_age = Window.partitionBy('Department').orderBy('Age')
df_ranked = df_adv.withColumn('Age Rank', F.rank().over(w_dept_age))

# Broadcast the department average age back to every employee row,
# then flag who is above their department average
w_dept_avg = Window.partitionBy('Department')
df_ranked = (
    df_ranked
    .withColumn('Dept Avg Age', F.round(F.avg('Age').over(w_dept_avg), 1))
    .withColumn('Above Dept Avg', F.col('Age') > F.col('Dept Avg Age'))
)
display(df_ranked)

# Rank departments by average tenure (dense_rank over the aggregated result)
dept_ranked = dept_stats.withColumn(
    'Tenure Rank',
    F.dense_rank().over(Window.orderBy(F.desc('Avg Tenure')))
)
display(dept_ranked)

In [0]:
# Convert Joining Year into a real date and derive time features
df_dated = (
    df_ranked
    .withColumn(
        'Joining Date',
        F.to_date(F.concat(F.col('Joining Year').cast('string'), F.lit('-01-01')))
    )
    .withColumn('Joining Quarter', F.quarter('Joining Date'))
)

# Veterans: 10+ years of service, longest-serving first
veterans = (
    df_dated
    .filter(F.col('Years of Service') >= 10)
    .orderBy(F.asc('Joining Date'))
)
display(veterans)